In [28]:
%pip install pyampute
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [29]:
import importlib
import missing_data_analysis as missing_data_analysis
importlib.reload(missing_data_analysis)
from missing_data_analysis import analyze_and_impute

In [30]:
"""
NHANES Data Pull — Hypertension Risk Prediction
=================================================
Fetches individual-level NHANES data directly from CDC servers via HTTPS.
Pulls 9 component files, merges on participant ID (SEQN),
cleans the result, and exports to CSV.

Target variable: hypertension_risk
  1 = at risk (diagnosed hypertensive OR measured BP >= 130/80 mmHg)
  0 = not at risk

Features:
  Demographics  — age, sex, race, education, income
  Blood pressure— systolic & diastolic readings (avg of 3 measurements)
  Body measures — BMI, waist circumference
  Smoking       — ever smoker, current smoker
  Alcohol       — drinking frequency, avg drinks/day
  Physical act. — vigorous/moderate work & recreational activity
  Diet          — sodium intake, total calories
  Lab values    — total cholesterol, fasting glucose

NHANES 2021-2023 cycle (most recent available).
No API key required — data fetched directly from:
  https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/<filename>.XPT
"""

import requests
import pandas as pd
import numpy as np
from io import BytesIO
from functools import reduce
import os
import sklearn
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
from missing_data_analysis import analyze_and_impute

# ── Config ───────────────────────────────────────────────────────────────────
CYCLE      = "2021"
BASE_URL   = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{CYCLE}/DataFiles/"

# (filename, friendly label)
COMPONENTS = [
    ("DEMO_L",  "Demographics"),
    ("BPXO_L",   "Blood Pressure"),        # clinically measured BP
    ("BPQ_L",   "Blood Pressure Questionnaire"),  # diagnosed hypertension, high cholesterol, medication use
    ("BMX_L",   "Body Measures"),
    ("SMQ_L",   "Smoking"),
    ("ALQ_L",   "Alcohol"),
    ("PAQ_L",   "Physical Activity"),
    ("TCHOL_L", "Total Cholesterol"),     # lab value
    ("HDL_L",   "HDL Cholesterol"),       # lab value
    ("GLU_L",   "Fasting Glucose"),       # lab value
    ("TRIGLY_L", "Triglycerides & LDL"),  # lab values
]


# ── 1. Fetch XPT file from CDC ───────────────────────────────────────────────
def fetch_xpt(filename: str, label: str) -> pd.DataFrame:
    url = f"{BASE_URL}/{filename}.XPT"
    print(f"  Fetching {label} ({filename}.XPT)...")
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        df = pd.read_sas(BytesIO(resp.content), format="xport")
        print(f"    → {len(df):,} rows, {df.shape[1]} columns")
        return df
    except requests.exceptions.RequestException as e:
        print(f"    ERROR fetching {filename}: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"    ERROR reading {filename}: {e}")
        return pd.DataFrame()


# ── 2. Select relevant columns ───────────────────────────────────────────────
COLUMN_MAP = {
    "DEMO_L": [
        "SEQN",
        "RIDAGEYR",    # age in years
        "RIAGENDR",    # sex: 1=male, 2=female
        "RIDRETH3",    # race/ethnicity
        "DMDEDUC2",    # education level
        "INDFMPIR",    # poverty-income ratio
        "DMDMARTZ",    # marital status
    ],
    "BPXO_L": [
        "SEQN",
        # Systolic readings (3 measurements)
        "BPXOSY1", "BPXOSY2", "BPXOSY3",
        # Diastolic readings (3 measurements)
        "BPXODI1", "BPXODI2", "BPXODI3",
        # Pulse readings (3 measurements) — not used for target but may be predictive
        "BPXOPLS1", "BPXOPLS2", "BPXOPLS3",
    ],
    "BPQ_L": [
        "SEQN",
        "BPQ020",      # ever told have high BP: 1=yes, 2=no
        "BPQ030",      # told twice+: 1=yes, 2=no
        "BPQ101D",     # taking prescription for high BP: 1=yes, 2=no,
        "BPQ080",      # told high cholesterol: 1=yes, 2=no
        "BPQ150",      # taking prescription for high cholesterol: 1=yes, 2=no
    ],
    "BMX_L": [
        "SEQN",
        "BMXBMI",      # BMI (kg/m²)
        "BMXWAIST",    # waist circumference (cm)
        "BMXWT",       # weight (kg)
        "BMXHT",       # height (cm)
    ],
    "SMQ_L": [
        "SEQN",
        "SMQ020",      # smoked 100+ cigarettes in life: 1=yes, 2=no
        "SMQ040",      # smoke now: 1=every day, 2=some days, 3=not at all
    ],
    "ALQ_L": [
        "SEQN",
        "ALQ111",      # drank ever: 1=yes, 2=no
        "ALQ121",      # how often drank past year (0=never, 10=every day)
        "ALQ130",      # avg drinks on days drank
    ],
    "PAQ_L": [
        "SEQN",
        "PAD680",      # time spent sitting on a typical day (minutes)
        "PAD790Q",     # moderate work frequency
        "PAD790U",     # moderate work frequency units (d/w/m/y)
        "PAD800",      # moderate activity duration (minutes)
        "PAD810Q",     # vigorous work frequency
        "PAD810U",     # vigorous work frequency units (d/w/m/y)
        "PAD820",      # vigorous activity duration (minutes)
    ],
    "TCHOL_L": [
        "SEQN",
        "LBXTC",       # total cholesterol (mg/dL)
    ],
    "HDL_L": [
        "SEQN",
        "LBDHDD",      # HDL cholesterol (mg/dL)
    ],
    "GLU_L": [
        "SEQN",
        "LBXGLU",      # fasting glucose (mg/dL)
    ],
    "TRIGLY_L": [
        "SEQN",
        "LBDLDL",      # LDL cholesterol (mg/dL) - calculated by Friedewald equation, LBDLDL = LBXTC - LBDHDD - LBXTR/5
        "LBDLDLM",     # LDL cholesterol (mg/dL) - Martin-Hopkins method, LBDLDLM = (LBXTC/0.948 – LBDHDD/0.971 – (LBXTR/8.56 + (LBXTR * (LBXTC – LBDHDD))/2140 – LBXTR^2/16100) – 9.44)
        "LBDLDLN",     # LDL cholesterol (mg/dL) - NIH equation, LBDLDLN = 0.9955 * (LBXTC – LBDHDD – LBXTR/5)
],
    
}

def select_columns(frames: dict) -> dict:
    selected = {}
    for name, df in frames.items():
        if df.empty:
            selected[name] = df
            continue
        cols = [c for c in COLUMN_MAP.get(name, ["SEQN"]) if c in df.columns]
        selected[name] = df[cols]
        missing = set(COLUMN_MAP.get(name, [])) - set(df.columns)
        if missing:
            print(f"    Note: {name} missing columns: {missing}")
    return selected


# ── 3. Merge on SEQN ─────────────────────────────────────────────────────────
def merge_components(frames: dict) -> pd.DataFrame:
    valid = [df for df in frames.values() if not df.empty and "SEQN" in df.columns]
    if not valid:
        raise ValueError("No valid DataFrames to merge.")
    df = reduce(lambda a, b: a.merge(b, on="SEQN", how="inner"), valid)
    print(f"\nMerged shape: {df.shape}")
    return df


# ── 4. Engineer blood pressure and pulse averages ──────────────────────────────────────
def engineer_bpp(df: pd.DataFrame) -> pd.DataFrame:
    """
    Average the 3 readings.
    Standard clinical practice uses the average of readings 2 & 3
    (reading 1 is a baseline). We compute both for completeness.
    """
    sys_cols  = [c for c in ["BPXOSY1", "BPXOSY2", "BPXOSY3"] if c in df.columns]
    dia_cols  = [c for c in ["BPXODI1", "BPXODI2", "BPXODI3"] if c in df.columns]
    pulse_cols = [c for c in ["BPXOPLS1", "BPXOPLS2", "BPXOPLS3"] if c in df.columns]

    if sys_cols:
        df["systolic_avg"]  = df[sys_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        # Clinical avg: readings 2 & 3 only
        if "BPXOSY2" in df.columns and "BPXOSY3" in df.columns:
            df["systolic_clinical"] = df[["BPXOSY2", "BPXOSY3"]].apply(
                pd.to_numeric, errors="coerce").mean(axis=1)
        # Remove raw systolic columns to avoid confusion
        df = df.drop(columns=sys_cols, errors="ignore")

    if dia_cols:
        df["diastolic_avg"] = df[dia_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        if "BPXODI2" in df.columns and "BPXODI3" in df.columns:
            df["diastolic_clinical"] = df[["BPXODI2", "BPXODI3"]].apply(
                pd.to_numeric, errors="coerce").mean(axis=1)
        # Remove raw diastolic columns to avoid confusion
        df = df.drop(columns=dia_cols, errors="ignore")

    if pulse_cols:
        df["pulse_avg"] = df[pulse_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        if "BPXOPLS2" in df.columns and "BPXOPLS3" in df.columns:
            df["pulse_clinical"] = df[["BPXOPLS2", "BPXOPLS3"]].apply(
                pd.to_numeric, errors="coerce").mean(axis=1)
        # Remove raw pulse columns to avoid confusion
        df = df.drop(columns=pulse_cols, errors="ignore")

    return df


# ── 5. Build target variable ─────────────────────────────────────────────────
def build_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    hypertension_risk = 1 if ANY of:
      - Previously diagnosed with high blood pressure (BPQ020 == 1)
      - Currently on BP medication (BPQ101D == 1)
      - Measured systolic >= 130 mmHg (ACC/AHA 2017 guideline)
      - Measured diastolic >= 80 mmHg

    hypertension_risk = 0 if:
      - No diagnosis AND no medication AND BP < 130/80

    Rows where BP was not measured AND no diagnosis info → dropped.
    """
    conditions = []

    # Diagnosed hypertension
    if "BPQ020" in df.columns:
        diagnosed = pd.to_numeric(df["BPQ020"], errors="coerce") == 1
        conditions.append(diagnosed)

    # On BP medication
    if "BPQ101D" in df.columns:
        on_meds = pd.to_numeric(df["BPQ101D"], errors="coerce") == 1
        conditions.append(on_meds)

    # Measured BP >= 130/80 (ACC/AHA Stage 1 Hypertension threshold)
    if "systolic_clinical" in df.columns:
        high_sys = pd.to_numeric(df["systolic_clinical"], errors="coerce") >= 130
        conditions.append(high_sys)

    if "diastolic_clinical" in df.columns:
        high_dia = pd.to_numeric(df["diastolic_clinical"], errors="coerce") >= 80
        conditions.append(high_dia)

    if not conditions:
        raise ValueError("No columns available to build target variable.")

    # Print summary of conditions before combining
    print("\n── Target Conditions Summary ──────────────────────────────────────")
    print(pd.concat(conditions, axis=1))

    # At risk if ANY condition is true
    df["hypertension_risk"] = (
        pd.concat(conditions, axis=1).any(axis=1).astype("Int64")
    )

    # Drop rows where target cannot be determined (all conditions NaN)
    df = df.dropna(subset=["hypertension_risk"])
    df["hypertension_risk"] = df["hypertension_risk"].astype(int)

    return df


# ── 6. Clean & recode ─────────────────────────────────────────────────────────
def clean(df: pd.DataFrame) -> pd.DataFrame:
 
    # ── Step 1: Replace refused/don't know codes ──────────────────────────
    refused_codes = [7, 9, 77, 99, 777, 999, 7777, 9999]
    df = df.replace(refused_codes, np.nan)
 
    # ── Step 2: Demographic recoding ──────────────────────────────────────
 
    # Convert participant ID and age to integers
    if "SEQN" in df.columns:
        df["SEQN"] = pd.to_numeric(df["SEQN"], errors="coerce").astype("Int64")
    if "RIDAGEYR" in df.columns:
        df["RIDAGEYR"] = pd.to_numeric(df["RIDAGEYR"], errors="coerce").astype("Int64")
 
    # Convert sex to binary Male column
    if "RIAGENDR" in df.columns:
        df["sex"] = pd.to_numeric(df["RIAGENDR"], errors="coerce").map({1: 1, 2: 0}).astype("Int64")
        df = df.rename(columns={"sex": "male"})

        # Remove original RIAGENDR column to avoid confusion
        df = df.drop(columns=["RIAGENDR"])
 
    # Race/ethnicity labels
    if "RIDRETH3" in df.columns:
        df["RIDRETH3"] = pd.to_numeric(df["RIDRETH3"], errors="coerce").map({
            1: "Mexican American",
            2: "Other Hispanic",
            3: "Non-Hispanic White",
            4: "Non-Hispanic Black",
            6: "Non-Hispanic Asian",
            7: "Other"
        }).astype("category")
 
    # Education labels
    if "DMDEDUC2" in df.columns:
        df["DMDEDUC2"] = pd.to_numeric(df["DMDEDUC2"], errors="coerce").map({
            1: "<HS",
            2: "HS - no diploma",
            3: "HS/GED or equivalent",
            4: "Some college or AA degree",
            5: "College graduate or above",
        }).astype("category")
 
    # Marital status labels
    if "DMDMARTZ" in df.columns:
        df["DMDMARTZ"] = pd.to_numeric(df["DMDMARTZ"], errors="coerce").map({
            1: "Married",
            2: "Widowed",
            3: "Divorced",
            4: "Separated",
            5: "Never married",
            6: "Living with partner",
        }).astype("category")

    # Diagnosed high BP
    if "BPQ020" in df.columns:
        df["BPQ020"] = pd.to_numeric(df["BPQ020"], errors="coerce").map({1: 1, 2: 0})
    else:
        df["BPQ020"] = 0  

    # On BP medication labels
    if "BPQ101D" in df.columns:
        df["BPQ101D"] = pd.to_numeric(df["BPQ101D"], errors="coerce").map({1: "Yes", 2: "No"}).astype("category")

    # On Cholesterol medication labels
    if "BPQ150" in df.columns:
        df["BPQ150"] = pd.to_numeric(df["BPQ150"], errors="coerce").map({1: "Yes", 2: "No"}).astype("category")
 
    # ── Step 3: Logical consistency fixes (before imputation) ─────────────
 
    # If not diagnosed once, can't be diagnosed twice
    if "BPQ020" in df.columns and "BPQ030" in df.columns:
        df.loc[df["BPQ020"] == 0, "BPQ030"] = 0
 
    # If never smoked 100 cigarettes, smoke frequency = not at all
    if "SMQ020" in df.columns and "SMQ040" in df.columns:
        df.loc[df["SMQ020"] == 2, "SMQ040"] = 3
 
    # If never drank, drink frequency = 0, avg drinks = 0
    if "ALQ111" in df.columns and "ALQ121" in df.columns:
        df.loc[df["ALQ111"] == 2, "ALQ121"] = 0
    if "ALQ111" in df.columns and "ALQ130" in df.columns:
        df.loc[df["ALQ111"] == 2, "ALQ130"] = 0
 
    # If moderate activity frequency is 0, duration should be 0
    if "PAD790Q" in df.columns and "PAD800" in df.columns:
        df.loc[pd.to_numeric(df["PAD790Q"], errors="coerce").abs() <= 1e-10, "PAD800"] = 0
 
    # If vigorous activity frequency is 0, duration should be 0
    if "PAD810Q" in df.columns and "PAD820" in df.columns:
        df.loc[pd.to_numeric(df["PAD810Q"], errors="coerce").abs() <= 1e-10, "PAD820"] = 0

 
    # ── Step 4: Derived features — smoking, alcohol, cholesterol ──────────
 
    if "SMQ020" in df.columns:
        df["ever_smoker"] = pd.to_numeric(df["SMQ020"], errors="coerce").map({1: 1, 2: 0})
    else:
        df["ever_smoker"] = 0
 
    if "SMQ040" in df.columns:
        smq040_numeric = pd.to_numeric(df["SMQ040"], errors="coerce")
        df["current_smoker"] = (
            (df["ever_smoker"] == 1) & (smq040_numeric.isin([1]))
        ).astype(int)
    else:
        df.loc[df["ever_smoker"] != 1, "current_smoker"] = 0
 
    if "ALQ111" in df.columns:
        df["drinks_alcohol"] = pd.to_numeric(df["ALQ111"], errors="coerce").map({1: 1, 2: 0})
        # Remove original ALQ111 column to avoid confusion
        df = df.drop(columns=["ALQ111"])
    
    if "ALQ121" in df.columns:
        df["drink_frequency_past_year"] = pd.to_numeric(df["ALQ121"], errors="coerce").map({
            0: 0,
            1: 365,
            2: 300,
            3: 208,
            4: 104,
            5: 52,
            6: 24,
            7: 12,
            8: 8,
            9: 4,
            10:1
            })
        # Remove original ALQ121 column to avoid confusion
        df = df.drop(columns=["ALQ121"])
 
    if "BPQ080" in df.columns:
        df["high_cholesterol"] = pd.to_numeric(df["BPQ080"], errors="coerce").map({1: 1, 2: 0})
        # Remove original BPQ080 column to avoid confusion
        df = df.drop(columns=["BPQ080"])
 
    # ── Step 5: Compute activity minutes per week ──────────────────────────
    unit_to_weekly = {
        b'D': 7,
        b'W': 1,
        b'M': 1 / 4.345,
        b'Y': 1 / 52.18,
        b'':  None,
    }
 
    if "PAD790Q" in df.columns and "PAD790U" in df.columns and "PAD800" in df.columns:
        moderate_freq = pd.to_numeric(df["PAD790Q"], errors="coerce")
        moderate_freq = moderate_freq.where(moderate_freq.abs() > 1e-10, 0)
        moderate_unit = df["PAD790U"].map(unit_to_weekly)
        moderate_mins = pd.to_numeric(df["PAD800"], errors="coerce")
        df["moderate_minutes_per_week"] = moderate_freq * moderate_unit * moderate_mins
        # Remove original columns to avoid confusion
        df = df.drop(columns=["PAD790Q", "PAD790U", "PAD800"])
 
    if "PAD810Q" in df.columns and "PAD810U" in df.columns and "PAD820" in df.columns:
        vigorous_freq = pd.to_numeric(df["PAD810Q"], errors="coerce")
        vigorous_freq = vigorous_freq.where(vigorous_freq.abs() > 1e-10, 0)
        vigorous_unit = df["PAD810U"].map(unit_to_weekly)
        vigorous_mins = pd.to_numeric(df["PAD820"], errors="coerce")
        df["vigorous_minutes_per_week"] = vigorous_freq * vigorous_unit * vigorous_mins
        # Remove original columns to avoid confusion
        df = df.drop(columns=["PAD810Q", "PAD810U", "PAD820"])
    
    # ── Step 6: Rename columns ─────────────────────────────────────────────
    rename = {
        "SEQN":      "participant_id",
        "RIDAGEYR":  "age",
        "RIAGENDR":  "sex",
        "RIDRETH3":  "race_ethnicity",
        "DMDEDUC2":  "education",
        "INDFMPIR":  "poverty_income_ratio",
        "DMDMARTZ":  "marital_status",
        "BMXBMI":    "bmi",
        "BMXWAIST":  "waist_cm",
        "BMXWT":     "weight_kg",
        "BMXHT":     "height_cm",
        "BPQ020":    "diagnosed_high_bp",
        "BPQ030":    "diagnosed_twice",
        "BPQ101D":   "on_chol_medication",
        "BPQ080":    "diagnosed_high_chol",
        "BPQ150":    "on_bp_medication",
        "SMQ020":    "smoked_100_cigarettes",
        "SMQ040":    "smoke_frequency",
        "ALQ111":    "ever_drank",
        "ALQ121":    "drink_frequency_past_year",
        "ALQ130":    "avg_drinks_per_day",
        "LBXTC":     "total_cholesterol_mgdl",
        "LBXGLU":    "fasting_glucose_mgdl",
        "LBDHDD":    "HDL_cholesterol_mgdl",
        "LBDLDL":    "LDL_cholesterol_mgdl_friedewald",
        "LBDLDLM":   "LDL_cholesterol_mgdl_martin",
        "LBDLDLN":   "LDL_cholesterol_mgdl_nih",
        "BPXOPLS1":  "pulse",
        "PAD680":    "time_sitting",
        "PAD790Q":   "moderate_work_frequency",
        "PAD790U":   "moderate_work_frequency_units",
        "PAD800":    "moderate_activity_duration",
        "PAD810Q":   "vigorous_work_frequency",
        "PAD810U":   "vigorous_work_frequency_units",
        "PAD820":    "vigorous_activity_duration",
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    
    # ── Step 7: Compute physically active flag using imputed values ────────────────────────
    mod_active = df["moderate_minutes_per_week"] >= 150
    vig_active = df["vigorous_minutes_per_week"] >= 75
    df["physically_active"] = (mod_active | vig_active).astype(int)

    # Save file with renamed columns before imputation for easier analysis of missingness patterns

    out_path = "../data/nhanes_hypertension_risk_pre_imputation.csv"
    df.to_csv(out_path, index=False)
    print(f"\n✓ Saved {len(df):,} records → {out_path}")
    
    # ── Step 8: Missingness check and Imputation ────────────────────────
    df = analyze_and_impute(df, threshold=0.0, knn_neighbors=3, top_n_context=3)
 
    print(f"\nShape after cleaning: {df.shape}")
    return df
        
# ── 7. Summary ────────────────────────────────────────────────────────────────
def summarize(df: pd.DataFrame):
    print("\n── Feature Columns ──────────────────────────────────────────")
    print(df.dtypes.to_string())

    print("\n── Missing Values ───────────────────────────────────────────")
    # Count missing values per column and percentage of total rows
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    missing = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": (missing / len(df) * 100).round(1)
    })
    if missing.empty:
        print("  None")
    else:
        print(missing.to_string())

    if "hypertension_risk" in df.columns:
        counts = df["hypertension_risk"].value_counts()
        total  = len(df)
        print(f"\n── Target Distribution ──────────────────────────────────────")
        print(f"  Not at risk (0): {counts.get(0, 0):,}  "
              f"({counts.get(0, 0)/total*100:.1f}%)")
        print(f"  At risk     (1): {counts.get(1, 0):,}  "
              f"({counts.get(1, 0)/total*100:.1f}%)")

    if "systolic_clinical" in df.columns and "diastolic_clinical" in df.columns:
        print(f"\n── Blood Pressure Summary ───────────────────────────────────")
        print(f"  Systolic  — mean: {df['systolic_clinical'].mean():.1f}  "
              f"std: {df['systolic_clinical'].std():.1f}")
        print(f"  Diastolic — mean: {df['diastolic_clinical'].mean():.1f}  "
              f"std: {df['diastolic_clinical'].std():.1f}")

    print(f"\n── Sample Rows ──────────────────────────────────────────────")
    print(df.head(3))


# ── Main ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print(f"NHANES {CYCLE} — Hypertension Risk Dataset")
    print("=" * 60)

    # Step 1: Fetch
    print("\n[1/5] Fetching XPT files from CDC servers...")
    frames = {}
    for filename, label in COMPONENTS:
        frames[filename] = fetch_xpt(filename, label)

    # Step 2: Select columns
    print("\n[2/5] Selecting relevant columns...")
    frames = select_columns(frames)

    # Step 3: Merge
    print("\n[3/5] Merging on participant ID (SEQN)...")
    df = merge_components(frames)
    
    # Print head and summary of preliminary merged dataset before engineering features and cleaning
    print(f"\nPreliminary merged dataset shape: {df.shape}")
    print(df.head(3))
    info = df.info()

    # Step 4: Engineer BP averages + target
    print("\n[4/5] Engineering blood pressure features & target variable...")
    df = engineer_bpp(df)
    df = build_target(df)

    # Print head and summary of merged dataset after engineering features
    print(f"\nMerged dataset shape after engineering features: {df.shape}")
    print(df.head(3))
    df.info()

    # Step 5: Clean
    print("\n[5/5] Cleaning and recoding...")
    df = clean(df)

    # Print head and summary of final cleaned dataset
    print(f"\nFinal cleaned dataset shape: {df.shape}")
    print(df.head(3))
    print("\nSummary statistics:")
    print(df.describe())

    # Create output directory if it doesn't exist
    os.makedirs("../data", exist_ok=True)
    
    # Save
    out_path = "../data/nhanes_hypertension_risk.csv"
    df.to_csv(out_path, index=False)
    print(f"\n✓ Saved {len(df):,} records → {out_path}")


NHANES 2021 — Hypertension Risk Dataset

[1/5] Fetching XPT files from CDC servers...
  Fetching Demographics (DEMO_L.XPT)...
    → 11,933 rows, 27 columns
  Fetching Blood Pressure (BPXO_L.XPT)...
    → 7,801 rows, 12 columns
  Fetching Blood Pressure Questionnaire (BPQ_L.XPT)...
    → 8,501 rows, 6 columns
  Fetching Body Measures (BMX_L.XPT)...
    → 8,860 rows, 22 columns
  Fetching Smoking (SMQ_L.XPT)...
    → 9,015 rows, 9 columns
  Fetching Alcohol (ALQ_L.XPT)...
    → 6,337 rows, 9 columns
  Fetching Physical Activity (PAQ_L.XPT)...
    → 8,153 rows, 8 columns
  Fetching Total Cholesterol (TCHOL_L.XPT)...
    → 8,068 rows, 4 columns
  Fetching HDL Cholesterol (HDL_L.XPT)...
    → 8,068 rows, 4 columns
  Fetching Fasting Glucose (GLU_L.XPT)...
    → 3,996 rows, 4 columns
  Fetching Triglycerides & LDL (TRIGLY_L.XPT)...
    → 3,996 rows, 10 columns

[2/5] Selecting relevant columns...

[3/5] Merging on participant ID (SEQN)...

Merged shape: (3562, 43)

Preliminary merged dataset